In [29]:
import pickle
import torch
import time
import copy
import numpy as np
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, AdamW
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

In [22]:
with open(f"/content/drive/MyDrive/dataset/train.pickle", "rb") as file:
    train_data = pickle.load(file)

with open(f"/content/drive/MyDrive/dataset/validation.pickle", "rb") as file:
    val_data = pickle.load(file)

with open(f"/content/drive/MyDrive/dataset/test.pickle", "rb") as file:
    test_data = pickle.load(file)

In [ ]:
train_data

{'tokens': array([list(['WHEAT', 'SALES', '-', 'Taiwan', 'Flour', 'Mills', 'Assn', 'bought', '98,000', 'tonnes', 'of', 'U.S.', 'No.1', 'or', 'No.2', 'wheat', 'from', 'Cargill', ',', 'Mitsui', ',', 'Continental', 'and', 'Louis', 'Dreyfus', 'Corp', 'for', 'shipment', 'from', 'the', 'Pacific', 'Northwest', '.']),
        list(['CYCLING', '-', 'SORENSEN', 'WINS', 'FOURTH', 'STAGE', 'OF', 'TOUR', 'OF', 'NETHERLANDS', '.']),
        list(['--', 'Greg', 'Frost', ',', '816', '561-8671']), ...,
        list(['Silva', 'excused', 'his', 'absence', 'from', 'Brazil', "'s", 'game', 'against', 'Russia', ',', 'on', 'Wednesday', ',', 'and', 'Saturday', "'s", 'match', 'with', 'the', 'Netherlands', 'by', 'saying', 'he', 'had', 'lost', 'his', 'passport', '.']),
        list(['44.', 'Bulgaria', '7.73']),
        list(['"', 'The', 'current', 'account', 'deficit', 'of', 'around', 'two', 'percent', 'of', 'GDP', 'is', 'a', 'sustainable', 'level', 'of', 'deficit', 'given', 'the', 'expected', 'real', 'growth', '

In [3]:
bert_model_name = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
print(f"length train_data [0]: {len(train_data['tokens'][0])}")
print(f"length tokenized train_data [0]: {len(tokenizer(train_data['tokens'][0] , is_split_into_words=True)['input_ids'])}")

length train_data [0]: 33
length tokenized train_data [0]: 58


In [ ]:
def align_tags(tokenized_input, tags):
    word_ids = tokenized_input.word_ids()
    previous_word_idx = None
    tags_ids = []

    for word_idx in word_ids:
        if word_idx is None or word_idx == previous_word_idx:
            tags_ids.append(-1)
        else:
            tags_ids.append(tags[word_idx] if word_idx < len(tags) else -1)

        previous_word_idx = word_idx

    return tags_ids

In [ ]:
print(align_tags(tokenizer(train_data['tokens'][0] , is_split_into_words=True) , train_data['ner_tags'][0]))

[-1, 0, -1, -1, -1, 0, -1, -1, 0, 3, 4, -1, 4, 4, -1, -1, 0, 0, -1, -1, 0, 0, 5, -1, -1, -1, 0, -1, -1, 0, 0, -1, -1, 0, 0, 3, -1, 0, 3, -1, -1, 0, 3, 0, 3, 4, -1, -1, -1, 4, 0, 0, 0, 0, 5, 6, 0, -1]


In [ ]:
print(train_data['tokens'][0])

['WHEAT', 'SALES', '-', 'Taiwan', 'Flour', 'Mills', 'Assn', 'bought', '98,000', 'tonnes', 'of', 'U.S.', 'No.1', 'or', 'No.2', 'wheat', 'from', 'Cargill', ',', 'Mitsui', ',', 'Continental', 'and', 'Louis', 'Dreyfus', 'Corp', 'for', 'shipment', 'from', 'the', 'Pacific', 'Northwest', '.']


In [ ]:
print(val_data['ner_tags'])
print(val_data['pos_tags'])

[list([0, 0, 0, 5, 0, 3, 4, 0, 0, 0]) list([5, 0, 0])
 list([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]) ... list([3, 0, 0, 0, 0, 0])
 list([0, 0, 7, 8, 0, 0, 0, 0, 0, 0]) list([0, 1, 2, 2, 0, 5, 0, 0])]
[list([24, 22, 8, 22, 8, 22, 22, 22, 11, 7]) list([22, 16, 21])
 list([12, 21, 16, 21, 42, 40, 15, 22, 11, 7]) ...
 list([22, 11, 11, 11, 11, 8])
 list([22, 8, 22, 22, 22, 22, 22, 27, 24, 7])
 list([11, 22, 22, 22, 4, 22, 5, 11])]


In [ ]:
def preprocess_data(data, tokenizer):
    processed_data = {'tokens': [], 'ner_tags': [], 'pos_tags': []}
    for i in range(len(data['tokens'])):
        tokenized_input = tokenizer(data['tokens'][i], is_split_into_words=True)
        input_ids = tokenized_input['input_ids']
        ner_tags = align_tags(tokenized_input, data['ner_tags'][i])
        pos_tags = align_tags(tokenized_input, data['pos_tags'][i])

        processed_data['tokens'].append(input_ids)
        processed_data['ner_tags'].append(ner_tags)
        processed_data['pos_tags'].append(pos_tags)
    return processed_data

train_data_correct_tags = preprocess_data(train_data, tokenizer)
val_data_correct_tags = preprocess_data(val_data, tokenizer)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data):
        self.tokens = data['tokens']
        self.ner_tags = data['ner_tags']
        self.pos_tags = data['pos_tags']

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return {'input_ids': torch.tensor(self.tokens[idx], dtype=torch.long),
                'ner_tags': torch.tensor(self.ner_tags[idx], dtype=torch.long),
                'pos_tags': torch.tensor(self.pos_tags[idx], dtype=torch.long)}


In [ ]:
def collate_fn(batch):
    input_ids = pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    ner_tags = pad_sequence([item['ner_tags'] for item in batch], batch_first=True, padding_value=-1)
    pos_tags = pad_sequence([item['pos_tags'] for item in batch], batch_first=True, padding_value=-1)

    return {'input_ids': input_ids, 'ner_tags': ner_tags, 'pos_tags': pos_tags}

In [ ]:
# Create DataLoaders
train_dataset = CustomDataset(train_data_correct_tags)
val_dataset = CustomDataset(val_data_correct_tags)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
def align_logits(tokenized_inputs):
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-1)
        elif word_idx != previous_word_idx:
            label_ids.append(1)
        else:
            label_ids.append(-1)

        previous_word_idx = word_idx

    return label_ids

In [8]:
class MultiTaskBERT(torch.nn.Module):
    def __init__(self, bert_model_name, num_pos_tags, num_ner_tags, dropout_rate=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.pos_classifier = torch.nn.Linear(self.bert.config.hidden_size, num_pos_tags)
        self.ner_classifier = torch.nn.Linear(self.bert.config.hidden_size, num_ner_tags)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, input_ids, attention_mask):
        bert_output = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(bert_output.last_hidden_state)
        pos_logits = self.pos_classifier(sequence_output)
        ner_logits = self.ner_classifier(sequence_output)
        return pos_logits, ner_logits

    def predict(self, tokens):
        encoding = self.tokenizer(tokens, is_split_into_words=True, return_tensors='pt')
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        label_ids = torch.tensor(align_logits(encoding), device=input_ids.device)

        pos_logits, ner_logits = self(input_ids, attention_mask)
        pos_logits_clean = pos_logits[0][label_ids != -1]
        ner_logits_clean = ner_logits[0][label_ids != -1]

        pos_pred = pos_logits_clean.argmax(dim=1).tolist()
        ner_pred = ner_logits_clean.argmax(dim=1).tolist()

        return pos_pred, ner_pred


In [ ]:
# Define the model, optimizer, and loss functions
model = MultiTaskBERT(bert_model_name, num_pos_tags= 47, num_ner_tags= 9)
optimizer = AdamW(model.parameters(), lr=5e-6)
pos_criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)
ner_criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


MultiTaskBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [ ]:
def train_model(model, dataloaders, optimizer, pos_criterion, ner_criterion, num_epochs=10):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = float('inf')

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs-1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_pos_corrects = 0
            running_ner_corrects = 0
            total_pos_samples = 0
            total_ner_samples = 0

            for batch in dataloaders[phase]:
                input_ids = batch['input_ids'].to(device)
                ner_tags = batch['ner_tags'].to(device)
                pos_tags = batch['pos_tags'].to(device)
                attention_mask = (input_ids != tokenizer.pad_token_id).to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    pos_logits, ner_logits = model(input_ids, attention_mask)
                    pos_loss = pos_criterion(pos_logits.view(-1, pos_logits.size(-1)), pos_tags.view(-1))
                    ner_loss = ner_criterion(ner_logits.view(-1, ner_logits.size(-1)), ner_tags.view(-1))
                    loss = (pos_loss + ner_loss) / 2

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * input_ids.size(0)

                # Calculate accuracy
                valid_pos_positions = (pos_tags != -1)
                pos_preds = pos_logits.argmax(dim=-1)[valid_pos_positions]
                pos_tags_filtered = pos_tags[valid_pos_positions]

                valid_ner_positions = (ner_tags != -1)
                ner_preds = ner_logits.argmax(dim=-1)
                ner_preds_filtered = ner_preds.view(-1)[valid_ner_positions.view(-1)]
                ner_tags_filtered = ner_tags.view(-1)[valid_ner_positions.view(-1)]

                running_pos_corrects += torch.sum(pos_preds == pos_tags_filtered).item()
                running_ner_corrects += torch.sum(ner_preds_filtered == ner_tags_filtered).item()

                total_pos_samples += pos_tags_filtered.size(0)
                total_ner_samples += ner_tags_filtered.size(0)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.4f}')

            if phase == 'train':
                epoch_pos_acc = running_pos_corrects / total_pos_samples
                epoch_ner_acc = running_ner_corrects / total_ner_samples
                print(f'{phase} Pos Accuracy: {epoch_pos_acc:.4f}, Ner Accuracy: {epoch_ner_acc:.4f}')
            else:
                val_pos_acc = running_pos_corrects / total_pos_samples
                val_ner_acc = running_ner_corrects / total_ner_samples
                print(f'{phase} Pos Accuracy: {val_pos_acc:.4f}, Ner Accuracy: {val_ner_acc:.4f}')

            if phase == 'val' and val_pos_acc < best_acc:
                best_acc = val_pos_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    print('Training complete')
    model.load_state_dict(best_model_wts)
    return model

# Training the model
dataloaders = {'train': train_dataloader, 'val': val_dataloader}
model = train_model(model, dataloaders, optimizer, pos_criterion, ner_criterion, num_epochs=25)


Epoch 0/24
----------
train Loss: 0.0246
train Pos Accuracy: 0.9876, Ner Accuracy: 0.9985
val Loss: 0.1148
val Pos Accuracy: 0.9628, Ner Accuracy: 0.9899
Epoch 1/24
----------
train Loss: 0.0213
train Pos Accuracy: 0.9893, Ner Accuracy: 0.9987
val Loss: 0.1200
val Pos Accuracy: 0.9621, Ner Accuracy: 0.9901
Epoch 2/24
----------
train Loss: 0.0193
train Pos Accuracy: 0.9903, Ner Accuracy: 0.9988
val Loss: 0.1202
val Pos Accuracy: 0.9621, Ner Accuracy: 0.9903
Epoch 3/24
----------
train Loss: 0.0175
train Pos Accuracy: 0.9913, Ner Accuracy: 0.9989
val Loss: 0.1232
val Pos Accuracy: 0.9622, Ner Accuracy: 0.9903
Epoch 4/24
----------
train Loss: 0.0156
train Pos Accuracy: 0.9921, Ner Accuracy: 0.9991
val Loss: 0.1259
val Pos Accuracy: 0.9632, Ner Accuracy: 0.9904
Epoch 5/24
----------
train Loss: 0.0142
train Pos Accuracy: 0.9927, Ner Accuracy: 0.9991
val Loss: 0.1293
val Pos Accuracy: 0.9632, Ner Accuracy: 0.9902
Epoch 6/24
----------
train Loss: 0.0129
train Pos Accuracy: 0.9935, Ner Acc

In [ ]:
import os

# Define a path to save the model
model_save_path = 'multi_task_bert_model.pth'

# Save the model state dictionary
torch.save(model.state_dict(), model_save_path)
print(f'Model saved to {model_save_path}')

Model saved to multi_task_bert_model.pth


In [4]:
bert_model_name = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

In [5]:
def preprocess_test_data(data, tokenizer):
    processed_data = {'tokens': []}
    for i in range(len(data['tokens'])):
        tokenized_input = tokenizer(data['tokens'][i], is_split_into_words=True)
        input_ids = tokenized_input['input_ids']
        processed_data['tokens'].append(input_ids)
    return processed_data

test_data_processed = preprocess_test_data(test_data, tokenizer)


In [6]:
class CustomTestDataset(Dataset):
    def __init__(self, data):
        self.tokens = data['tokens']

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return {'input_ids': torch.tensor(self.tokens[idx], dtype=torch.long)}

def collate_fn_test(batch):
    input_ids = pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return {'input_ids': input_ids, 'attention_mask': attention_mask}

test_dataset = CustomTestDataset(test_data_processed)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_test)


In [9]:
model = MultiTaskBERT(bert_model_name, num_pos_tags=47, num_ner_tags=9)
model.load_state_dict(torch.load("/content/drive/MyDrive/BERT-multitask/multi_task_bert_model.pth", map_location=torch.device('cpu')))
model.eval()


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

MultiTaskBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [10]:
# Load test data
with open(f"/content/drive/MyDrive/dataset/test.pickle", "rb") as file:
    test_data = pickle.load(file)

test_data_processed = preprocess_test_data(test_data, tokenizer)

In [11]:
# Create DataLoader for test data
class CustomTestDataset(Dataset):
    def __init__(self, data):
        self.tokens = data['tokens']

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return {'input_ids': torch.tensor(self.tokens[idx], dtype=torch.long)}

def collate_fn_test(batch):
    input_ids = pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return {'input_ids': input_ids, 'attention_mask': attention_mask}

test_dataset = CustomTestDataset(test_data_processed)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_test)

In [12]:
# Predict test data
def predict_test_data(model, dataloader):
    all_pos_preds = []
    all_ner_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']

            pos_logits, ner_logits = model(input_ids, attention_mask)

            valid_positions = (input_ids != tokenizer.pad_token_id)
            for i in range(input_ids.size(0)):
                pos_pred = pos_logits[i][valid_positions[i]].argmax(dim=-1).tolist()
                ner_pred = ner_logits[i][valid_positions[i]].argmax(dim=-1).tolist()
                all_pos_preds.append(pos_pred)
                all_ner_preds.append(ner_pred)

    return all_pos_preds, all_ner_preds

test_pos_preds, test_ner_preds = predict_test_data(model, test_dataloader)

In [18]:
test_data['tokens'][1]
tokenizer(test_data['tokens'][1], is_split_into_words=True).word_ids()

[None, 0, 1, 1, 1, 1, 2, 3, 4, 4, 4, 5, 6, 6, 6, None]

In [16]:
test_ner_preds[1]

[0, 3, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [19]:
test_data['tokens'][1]

['ST', 'LOUIS', '69', '65', '.515', '2', '1/2']

In [25]:
def align_predictions_to_tokens(tokens, pos_preds, ner_preds):
    aligned_pos_preds = []
    aligned_ner_preds = []

    for i in range(len(tokens)):
        word_ids = tokenizer(tokens[i], is_split_into_words=True).word_ids()
        aligned_pos_preds_example = []
        aligned_ner_preds_example = []

        # Track the last valid word id
        last_word_id = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                # Only keep predictions for the first subword of each word
                if word_id != last_word_id:
                    aligned_pos_preds_example.append(pos_preds[i][idx])
                    aligned_ner_preds_example.append(ner_preds[i][idx])
                last_word_id = word_id

        aligned_pos_preds.append(aligned_pos_preds_example)
        aligned_ner_preds.append(aligned_ner_preds_example)

    return aligned_pos_preds, aligned_ner_preds

# Align predictions to tokens
aligned_pos_preds, aligned_ner_preds = align_predictions_to_tokens(test_data['tokens'], test_pos_preds, test_ner_preds)

In [26]:
aligned_ner_preds[1]

[3, 4, 0, 0, 0, 0, 0]

In [27]:
test_pred = {'tokens': test_data["tokens"] , 'ner_tags': np.array(aligned_ner_preds, dtype=object) ,
             'pos_tags': np.array(aligned_pos_preds, dtype=object)}

In [28]:
# Saving the test predicted results
with open('test_pred.pickle', 'wb') as file:
  pickle.dump(test_pred, file)